# SPLADE: бенчмарк на SciFact (zero-shot, быстрый eval)

Отдельный блокнот рядом с `splade_experiments.ipynb`. **Обучение не запускается** —
берутся уже обученные на MS MARCO модели из `outputs/<version>/<run_id>/model/` и
прогоняются **zero-shot** по небольшому набору **SciFact** (BEIR).

## Зачем это нужно
Инференс на полном MS MARCO упирается в кодирование 8.8M пассажей и построение
индекса — это ~1.5 ч на каждую проверку. SciFact — это ~5K документов и ~300
запросов, индекс строится за секунды. Значит, сравнивать версии модели и смотреть,
растёт ли качество, можно почти мгновенно. SciFact — стандартный zero-shot бенчмарк
для SPLADE/BEIR, главная метрика — **nDCG@10**.

## Что не меняется
`splade_lab/` дополнен **только новыми** модулями (`datasets`, `modeling`,
`benchmark`, `stats`); существующие `data/model/train/evaluate/compare` и оба
прежних блокнота не тронуты. Кодирование и поиск переиспользуют те же функции
`evaluate.encode_sparse` / `evaluate.search`, что и обучение на MS MARCO.

## Порядок
1. **Setup** — окружение и устройство.
2. **Конфиг набора** — SciFact (одна строка; смена набора = другое имя).
3. **Данные** — скачиваются один раз, приводятся к канонической схеме TSV.
4. **Какие модели сравниваем** — выбор обученных прогонов из `outputs/`.
5. **Бенчмарк** — zero-shot прогон каждой модели по SciFact.
6. **Сводная таблица** — метрики всех моделей на SciFact.
7. **Значимость** — парные тесты (bootstrap + t-тест + Wilcoxon) по nDCG@10.
8. **Обоснование** — почему так сравнивать корректно.

## 1. Setup

In [1]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import json
import torch

from splade_lab.train import pick_device

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.yandex-team.ru/simple/

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
torch 2.5.1+cu121 | устройство: cuda (NVIDIA A100 80GB PCIe)


## 2. Конфиг набора (SciFact)

`datasets.beir_config("scifact")` возвращает готовый словарь источника: имя,
split (`test`), URL официального BEIR-зеркала и каталог на диске. **Чтобы сделать
блокнот под другой набор — поменяй только имя** (`"nfcorpus"`, `"scidocs"`,
`"arguana"`, ...). Всё остальное идентично.

In [2]:
from splade_lab import datasets

DATASET = "scifact"  # ГЛАВНЫЙ набор. Сменить набор = поменять только это имя.
DATA = datasets.beir_config(DATASET, num_eval_queries=-1)  # -1 = все запросы сплита

print(f"Набор: {DATA['name']} | split={DATA['split']}")
print(f"URL:   {DATA['url']}")
print(f"Каталог на диске: {datasets.dataset_dir(DATA)}")
print(f"Известные наборы: {sorted(datasets.BEIR_DATASETS)}")

Набор: scifact | split=test
URL:   https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip
Каталог на диске: /home/uvuv/workspace/sandbox_03/data/beir/scifact
Известные наборы: ['arguana', 'fiqa', 'nfcorpus', 'scidocs', 'scifact', 'trec-covid', 'webis-touche2020']


## 3. Данные (скачиваются один раз)

`prepare_beir` качает zip BEIR, распаковывает и приводит к той же схеме TSV, что
и MS MARCO (`collection.tsv` / `queries.tsv` / `qrels.tsv`). Поэтому те же
загрузчики `data.load_collection` / `data.load_queries` работают без изменений.
Идемпотентно: если TSV уже на месте — повторно ничего не качает.

In [3]:
ds_dir = datasets.prepare_beir(DATA)

print(f"\nКаталог: {ds_dir}")
print("Строк в файлах:", datasets.dataset_stats(ds_dir))

with open(ds_dir / "queries.tsv", encoding="utf-8") as f:
    print("\nПример запроса:  ", f.readline().strip()[:120])
with open(ds_dir / "collection.tsv", encoding="utf-8") as f:
    print("Пример документа:", f.readline().strip()[:120], "...")

[beir] scifact уже готов: /home/uvuv/workspace/sandbox_03/data/beir/scifact

Каталог: /home/uvuv/workspace/sandbox_03/data/beir/scifact
Строк в файлах: {'collection.tsv': 5183, 'queries.tsv': 300, 'qrels.tsv': 339}

Пример запроса:   1	0-dimensional biomaterials show inductive properties.
Пример документа: 4983	Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic re ...


## 4. Какие модели сравниваем

Модели берём из `outputs/<version>/` (последний по времени прогон каждой версии).
`MODELS` — список того, что сравнить: `version` (+ опц. `run_id`/`run_dir`,
`query_encoder`, `label`). Это и есть «легко подгружать разные модели».

Чтобы добавить новую версию в сравнение — допиши строку в `MODELS` (например
новую обученную модель `v3_*`). Ничего в коде менять не нужно.

In [4]:
from splade_lab import modeling

# Версии для сравнения. run_id не указан -> берётся самый свежий прогон версии.
MODELS = [
    {"version": "v1_splade_max", "label": "v1_splade_max"},
    {"version": "v2_splade_doc", "label": "v2_splade_doc"},
    # Пример фиксации конкретного прогона:
    # {"version": "v1_splade_max", "run_id": "20260613-005141", "label": "v1_old"},
]

for m in MODELS:
    try:
        run = modeling.resolve_model_run(version=m.get("version"),
                                         run_id=m.get("run_id"),
                                         run_dir=m.get("run_dir"))
        print(f"{m['label']:>16} -> {run}")
    except FileNotFoundError as e:
        print(f"{m['label']:>16} -> [нет модели] {e}")

   v1_splade_max -> /home/uvuv/workspace/sandbox_03/outputs/v1_splade_max/20260613-005141
   v2_splade_doc -> /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260613-180612


## 5. Бенчмарк (zero-shot прогон по SciFact)

`benchmark.run_benchmark` грузит сохранённую модель, кодирует корпус/запросы,
ищет top-k и считает метрики — агрегатно и по каждому запросу. Артефакты пишутся
в `outputs/benchmark/<dataset>/<label>__<run_id>/` (`summary.json`,
`per_query.csv`, `per_query.npz`) — отдельно от `outputs/<version>/`, чтобы не
мешать таблице обучения. Per-query метрики сохраняются для тестов значимости.

In [5]:
from splade_lab import benchmark

RECALL_KS = (10, 100)   # корпус SciFact ~5K, recall@1000 здесь не информативен
NDCG_KS = (10,)          # главная метрика BEIR

bench_results = {}
for m in MODELS:
    try:
        res = benchmark.run_benchmark(
            version=m.get("version"), run_id=m.get("run_id"),
            run_dir=m.get("run_dir"), query_encoder=m.get("query_encoder"),
            data_cfg=DATA, device=device,
            recall_ks=RECALL_KS, ndcg_ks=NDCG_KS,
            label=m["label"], save=True)
        bench_results[m["label"]] = res
    except (FileNotFoundError, RuntimeError) as e:
        print(f"[skip] {m['label']}: {e}")

for label, res in bench_results.items():
    agg = res["aggregate"]
    print(f"{label:>16}: nDCG@10={agg.get('ndcg@10'):.4f} "
          f"recall@10={agg.get('recall@10'):.4f} "
          f"| index={res['timing']['index_s']}s search={res['timing']['search_s']}s")

[bench] v1_splade_max (src=20260613-005141) на scifact qe=mlm device=cuda
[progress] лог прогресса: /home/uvuv/workspace/sandbox_03/outputs/logs/progress_20260614-225522.log


/home/uvuv/workspace/sandbox_03/splade_lab/evaluate.py:73: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  D = torch.sparse_csr_tensor(


[bench] scifact | mrr@10=0.5796 recall@10=0.7389 recall@100=0.8737 ndcg@10=0.6147 avg_nnz_doc=366.3025 avg_nnz_query=120.5233 | index=4.2s search=0.17s
[bench] артефакты: /home/uvuv/workspace/sandbox_03/outputs/benchmark/scifact/v1_splade_max__20260613-005141
[bench] v2_splade_doc (src=20260613-180612) на scifact qe=bow device=cuda
[bench] scifact | mrr@10=0.6028 recall@10=0.7639 recall@100=0.8963 ndcg@10=0.6369 avg_nnz_doc=249.4059 avg_nnz_query=18.8000 | index=4.09s search=0.12s
[bench] артефакты: /home/uvuv/workspace/sandbox_03/outputs/benchmark/scifact/v2_splade_doc__20260613-180612
   v1_splade_max: nDCG@10=0.6147 recall@10=0.7389 | index=4.2s search=0.17s
   v2_splade_doc: nDCG@10=0.6369 recall@10=0.7639 | index=4.09s search=0.12s


## 6. Сводная таблица метрик на SciFact

In [6]:
import pandas as pd

rows = []
for label, res in bench_results.items():
    agg = res["aggregate"]
    rows.append({
        "label": label,
        "version": res["version"],
        "run_id": res["run_id"],
        "query_encoder": res["query_encoder"],
        "ndcg@10": agg.get("ndcg@10"),
        "recall@10": agg.get("recall@10"),
        "recall@100": agg.get("recall@100"),
        "mrr@10": agg.get("mrr@10"),
        "avg_nnz_doc": agg.get("avg_nnz_doc"),
        "avg_nnz_query": agg.get("avg_nnz_query"),
        "n_eval_queries": agg.get("n_eval_queries"),
        "n_corpus_docs": agg.get("n_corpus_docs"),
        "index_s": res["timing"]["index_s"],
        "search_s": res["timing"]["search_s"],
    })

df = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
out_csv = benchmark.BENCH_OUTPUTS / DATASET / "comparison.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print(f"\n[bench] CSV: {out_csv}")
df

        label       version          run_id query_encoder  ndcg@10  recall@10  recall@100   mrr@10  avg_nnz_doc  avg_nnz_query  n_eval_queries  n_corpus_docs  index_s  search_s
v2_splade_doc v2_splade_doc 20260613-180612           bow 0.636886   0.763889    0.896333 0.602815   249.405943      18.800000             300           5183     4.09      0.12
v1_splade_max v1_splade_max 20260613-005141           mlm 0.614666   0.738889    0.873667 0.579570   366.302527     120.523333             300           5183     4.20      0.17

[bench] CSV: /home/uvuv/workspace/sandbox_03/outputs/benchmark/scifact/comparison.csv


,label,version,run_id,query_encoder,ndcg@10,recall@10,recall@100,mrr@10,avg_nnz_doc,avg_nnz_query,n_eval_queries,n_corpus_docs,index_s,search_s
0,v2_splade_doc,v2_splade_doc,20260613-180612,bow,0.636886,0.763889,0.896333,0.602815,249.405943,18.800000,300,5183,4.09,0.12
1,v1_splade_max,v1_splade_max,20260613-005141,mlm,0.614666,0.738889,0.873667,0.579570,366.302527,120.523333,300,5183,4.20,0.17


## 7. Статистическая значимость различий

Сравнивать средние мало: важно, **значимо** ли отличие или это шум. Метрику
считаем по каждому запросу и сравниваем версии **парно** (на одних и тех же
запросах SciFact). `stats.compare_pair` даёт сразу три теста по nDCG@10:

- **парный bootstrap** (10k ресэмплов) — Δ, 95% ДИ и p (стандарт в IR);
- **парный t-тест** — классический;
- **Wilcoxon signed-rank** — непараметрический.

Если все три дают p < 0.05 и ДИ не накрывает 0 — улучшение надёжно.

In [7]:
from splade_lab import stats

METRIC = "ndcg@10"
ALPHA = 0.05

# Базовая версия (с кем сравниваем остальных). По умолчанию — первая в MODELS.
labels = list(bench_results)
if len(labels) < 2:
    print("Нужно >=2 модели в bench_results для сравнения значимости.")
else:
    baseline = labels[0]
    pq_base = bench_results[baseline]["per_query"]
    comp_rows, pvals = [], {}
    for label in labels[1:]:
        cmp = stats.compare_pair(pq_base, bench_results[label]["per_query"],
                                 metric=METRIC, name_a=baseline, name_b=label,
                                 alpha=ALPHA, n_boot=10000, seed=0)
        pvals[f"{baseline}_vs_{label}"] = cmp["p_bootstrap"]
        comp_rows.append(cmp)
        print(f"\n=== {baseline}  vs  {label}  ({METRIC}) ===")
        print(f"  mean({baseline}) = {cmp['mean_a']:.4f} | mean({label}) = {cmp['mean_b']:.4f}")
        print(f"  Δ = {cmp['delta']:+.4f}  ({cmp['rel_improvement_pct']:+.1f}%)  "
              f"95% ДИ [{cmp['ci95_low']:+.4f}, {cmp['ci95_high']:+.4f}]")
        print(f"  p(bootstrap) = {cmp['p_bootstrap']:.4g} | "
              f"p(t-test) = {cmp['p_ttest']:.4g} | p(Wilcoxon) = {cmp['p_wilcoxon']:.4g}")
        print(f"  n_queries = {cmp['n_queries']} | значимо (bootstrap, α={ALPHA}): {cmp['significant']}")

    # Поправка на множественные сравнения (если версий больше двух).
    if len(pvals) > 1:
        print("\n=== Поправка Холма–Бонферрони (множественные сравнения) ===")
        for key, info in stats.holm_correction(pvals, alpha=ALPHA).items():
            print(f"  {key}: p={info['p']:.4g} -> p_adj={info['p_adj']:.4g} "
                  f"| отвергаем H0: {info['reject']}")


=== v1_splade_max  vs  v2_splade_doc  (ndcg@10) ===
  mean(v1_splade_max) = 0.6147 | mean(v2_splade_doc) = 0.6369
  Δ = +0.0222  (+3.6%)  95% ДИ [-0.0053, +0.0500]
  p(bootstrap) = 0.1176 | p(t-test) = 0.1163 | p(Wilcoxon) = 0.1326
  n_queries = 300 | значимо (bootstrap, α=0.05): False


## 8. Почему так сравнивать корректно (обоснование)

**1. Zero-shot на BEIR — стандартная практика.** SPLADE и другие модели,
обученные на MS MARCO, штатно оценивают на наборах BEIR (SciFact, NFCorpus и
т.д.) без дообучения. Это измеряет обобщающую способность представлений и не
требует обучающих данных целевого набора.

**2. Малый корпус ≠ потеря строгости.** SciFact (~5K докум., ~300 запросов)
индексируется за секунды, тогда как MS MARCO (8.8M) — ~1.5 ч. Меньший корпус
сокращает время прогона, но метрика nDCG@10 с qrels-градациями — та же, что в
публикациях; результаты на разных версиях сопоставимы между собой.

**3. Парные тесты убирают разброс запросов.** Разные запросы по-разному трудны.
Сравнивая две версии **на одних и тех же запросах** и тестируя попарные разницы
(а не независимые выборки), мы убираем межзапросную дисперсию — это резко
повышает чувствительность и является стандартом в IR.

**4. Три взаимодополняющих теста.** Bootstrap по запросам (непараметрический,
корректен на малых выборках; де-факто стандарт в IR), парный t-тест
(чувствителен при ~нормальных разницах) и Wilcoxon (устойчив к выбросам). Если
выводы согласованы — заключение надёжно. При сравнении многих версий применяем
поправку Холма–Бонферрони, чтобы контролировать ложные срабатывания.

**5. Ограничение.** Один маленький набор отражает узкий домен (научные
утверждения). Поэтому рядом лежат блокноты-копии для других наборов
(NFCorpus и т.д.): согласованный прирост на нескольких наборах — более сильное
свидетельство улучшения, чем на одном. Финальное число MS MARCO при желании
по-прежнему считается основным блокнотом — здесь же быстрая итеративная проверка.